In [1]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

In [2]:
def should_skip(script_name, data_name):
    """
    이미 결과 파일이 존재하는지 확인하는 함수
    경로 예시: result/AIR/test/trTFTF_AIR_mse_pred.csv
    """
    result_path = f"result/{data_name}/test"
    
    if os.path.exists(result_path):
        ## 해당 폴더 내의 파일 리스트를 가져옴
        files = os.listdir(result_path)

        ## 스크립트 이름으로 시작하는 파일이 5개 있으면 True 반환
        if sum([f.startswith(script_name) for f in files]) == 5:
            return True
        
    return False

def agg_nonskip(script_name, data_name):
    inference_path = f"inference"

    if os.path.exists(inference_path):
        files = os.listdir(inference_path)

        if sum([f.startswith(f"{script_name}_{data_name}") for f in files]) > 0:
            return True
        
    return False

In [3]:
datasets = ["TEM", "SOL", "ELE", "AIR", "WID", "MET", "coin"]
model_names = ['trTFTF', "trTFMLP", "trTFLSTM"]
num = 100  # Model k per loss

os.makedirs("inference", exist_ok = True)

In [4]:
for d in range(len(datasets)):
    for model_name in model_names:
        if should_skip(model_name, datasets[d]):
            #############################################################################################
            target_X = pd.read_csv(f"../data/{datasets[d]}/train_input_7.csv").iloc[:, 1:].values.astype(np.float32)
            target_y = pd.read_csv(f"../data/{datasets[d]}/train_output_7.csv").iloc[:, 1:].values.astype(np.float32)

            target_X_val = target_X[-round(target_X.shape[0] * 0.2):, :].astype(np.float32)
            target_y_val = target_y[-round(target_y.shape[0] * 0.2):].astype(np.float32)

            target_X = target_X[:-round(target_X.shape[0] * 0.2), :].astype(np.float32)
            target_y = target_y[:-round(target_y.shape[0] * 0.2)].astype(np.float32)
            test_X = pd.read_csv(f"../data/{datasets[d]}/val_input_7.csv").iloc[:, 1:].values.astype(np.float32)
            test_y = pd.read_csv(f"../data/{datasets[d]}/val_output_7.csv").iloc[:, 1:].values.astype(np.float32)

            #############################################################################################
            folder_path = f'result/{datasets[d]}/test/'
            folder_path2 = f'result/{datasets[d]}/val/'

            all_files = os.listdir(folder_path)
            all_files2 = os.listdir(folder_path2)
            print(f"--- 파일 검색 디버깅 ---")
            print(f"검색 중인 폴더: {folder_path}")
            print(f"폴더 안의 모든 파일: {all_files}")
            search_prefix = f"{model_name}_{datasets[d]}"
            print(f"찾고 있는 파일 시작 부분(prefix): '{search_prefix}'")
            print("--- 디버깅 끝 ---")
            files = sorted([f for f in all_files if f.startswith(f"{model_name}_{datasets[d]}") and f.endswith("pred.csv")])
            dataframes = [pd.read_csv(os.path.join(folder_path, f)) for f in files]

            files2 = sorted([f for f in all_files2 if f.startswith(f"{model_name}_{datasets[d]}") and f.endswith("pred.csv")])
            dataframes2 = [pd.read_csv(os.path.join(folder_path2, f)) for f in files2]

            all = np.array([np.stack(dataframes[i].iloc[:, 1:].values.reshape(num, -1, target_y.shape[1])) for i in range(len(dataframes))])
            all_val = np.array([np.stack(dataframes2[i].iloc[:, 1:].values.reshape(num, -1, target_y.shape[1])) for i in range(len(dataframes2))])
            #############################################################################################
            all_ens_lst, all_ens_rmse_lst = [], []
            bolt1_lst, bolt1_rmse_lst = [], []
            bolt3_lst, bolt3_rmse_lst = [], []
            bolt5_lst, bolt5_rmse_lst = [], []
            
            p = 10
            b = 50
            np.random.seed(100)  ### NEW: mean 평가 지표 기록용

            for i in range(b):
                # --------------------------------------------------------------
                # 1) 무작위 인덱스 샘플링
                # matrix = np.arange(100).reshape(10, 10)
                nums1 = random.sample(range(num), k=p)
                nums2 = random.sample(range(num), k=p)
                nums3 = random.sample(range(num), k=p)
                nums4 = random.sample(range(num), k=p)
                nums5 = random.sample(range(num), k=p)

                # nums1 = matrix[i]
                # nums2 = matrix[i]
                # nums3 = matrix[i]
                # nums4 = matrix[i]
                # nums5 = matrix[i]
                # --------------------------------------------------------------
                # 2) all_Ens (median 앙상블은 그대로)
                score = np.concatenate([all[0][nums1], all[1][nums2],
                                        all[2][nums3], all[3][nums4],
                                        all[4][nums5]], axis=0)

                all_ens = np.median(score, axis=0).flatten()
                all_ens_rmse = np.sqrt(mean_squared_error(test_y.flatten(), all_ens))
                all_ens_lst.append(all_ens)
                all_ens_rmse_lst.append(all_ens_rmse)

                ####################################################################
                mae_val = np.median(all_val[0][nums1], axis=0).flatten()  # .shape
                mape_val = np.median(all_val[1][nums2], axis=0).flatten()
                mase_val = np.median(all_val[2][nums3], axis=0).flatten()
                mse_val = np.median(all_val[3][nums4], axis=0).flatten()
                smape_val = np.median(all_val[4][nums5], axis=0).flatten()

                rmse_mae_val = np.sqrt(mean_squared_error(target_y_val.flatten(), mae_val))
                rmse_mape_val = np.sqrt(mean_squared_error(target_y_val.flatten(), mape_val))
                rmse_mase_val = np.sqrt(mean_squared_error(target_y_val.flatten(), mase_val))
                rmse_mse_val = np.sqrt(mean_squared_error(target_y_val.flatten(), mse_val))
                rmse_smape_val = np.sqrt(mean_squared_error(target_y_val.flatten(), smape_val))

                fin_pred_mae = np.median(all[0][nums1], axis=0).flatten()
                fin_pred_mape = np.median(all[1][nums2], axis=0).flatten()
                fin_pred_mase = np.median(all[2][nums3], axis=0).flatten()
                fin_pred_mse = np.median(all[3][nums4], axis=0).flatten()
                fin_pred_smape = np.median(all[4][nums5], axis=0).flatten()

                performance = np.array([rmse_mae_val, rmse_mape_val, rmse_mase_val, rmse_mse_val, rmse_smape_val])

                for beta in [1, 3, 5]:
                    bolt_rmse_lst_ = []
                    bolt = []
                    
                    weights = np.exp(-beta * performance)
                    gd = np.concatenate([fin_pred_mae.flatten().reshape(1, -1),
                                        fin_pred_mape.flatten().reshape(1, -1),
                                        fin_pred_mase.flatten().reshape(1, -1),
                                        fin_pred_mse.flatten().reshape(1, -1),
                                        fin_pred_smape.flatten().reshape(1, -1)], axis=0)

                    normalized_weights = weights / np.sum(weights)

                    # 각 모델의 예측값에 가중치를 부여하여 앙상블 예측 생성
                    ensemble_prediction = np.dot(normalized_weights, gd)
                    bolt.append(ensemble_prediction)
                    bolt_rmse = np.sqrt(mean_squared_error(test_y.flatten(), ensemble_prediction.flatten()))
                    bolt_rmse_lst_.append(bolt_rmse)

                    if beta == 1:
                        bolt1_lst.append(bolt)
                        bolt1_rmse_lst.append(bolt_rmse_lst_)
                    elif beta == 3:
                        bolt3_lst.append(bolt)
                        bolt3_rmse_lst.append(bolt_rmse_lst_)
                    else:
                        bolt5_lst.append(bolt)
                        bolt5_rmse_lst.append(bolt_rmse_lst_)

                # --------------------------------------------------------------

            a1 = np.min(np.array(bolt1_rmse_lst), axis=0)
            a3 = np.min(np.array(bolt3_rmse_lst), axis=0)
            a5 = np.min(np.array(bolt5_rmse_lst), axis=0)

            summary = pd.DataFrame({'All Ens': all_ens_rmse_lst,
                                    'Exp(1)': np.array(bolt1_rmse_lst).T[np.argmin(a1)],
                                    'Exp(3)': np.array(bolt3_rmse_lst).T[np.argmin(a3)],
                                    'Exp(5)': np.array(bolt5_rmse_lst).T[np.argmin(a5)]})
            summary.to_csv(f'inference/{model_name}_{datasets[d]}_RMSE_summary.csv')
            summary.describe()

            ## ============= MAE =============
            p = 10
            b = 30  ## 여긴 왜 30이야?
            all_ens_mae_lst = []
            bolt1_mae_lst = []
            bolt3_mae_lst = []
            bolt5_mae_lst = []
            mean_mae_lst = []

            bolt_all_data = []
            performance_lst = []

            for i in range(b):
                nums = [np.random.choice(num, size=p, replace=False) for _ in range(5)]

                score = np.concatenate([all[j][nums[j]] for j in range(5)], axis=0)
                all_ens = np.median(score, axis=0).flatten()
                all_ens_mae = mean_absolute_error(test_y.flatten(), all_ens)
                all_ens_mae_lst.append(all_ens_mae)

                mean_mae = np.mean(score, axis=0).flatten()
                all_mean_mae = mean_absolute_error(test_y.flatten(), mean_mae)
                mean_mae_lst.append(all_mean_mae)

                # 각 지표별 예측
                val_preds = [np.median(np.nan_to_num(all_val[j][nums[j]], nan=0), axis=0).flatten() for j in range(5)]
                perf_maes = [mean_absolute_error(target_y_val.flatten(), pred) for pred in val_preds]

                fin_preds = [np.median(np.nan_to_num(all[j][nums[j]], nan=0), axis=0).flatten() for j in range(5)]
                performance = np.array(perf_maes)
                performance_lst.append(performance)

                for beta in [1, 3, 5]:
                    bolt = []
                    bolt_mae_lst_ = []

                    weights = np.exp(-beta * performance)
                    normalized_weights = weights / np.sum(weights)
                    gd = np.stack(fin_preds, axis=0)
                    ensemble_prediction = np.dot(normalized_weights, gd)
                    bolt.append(ensemble_prediction)
                    bolt_mae = mean_absolute_error(test_y.flatten(), ensemble_prediction.flatten())
                    bolt_mae_lst_.append(bolt_mae)

                    if beta == 1:
                        bolt1_mae_lst.append(bolt_mae_lst_)
                    elif beta == 3:
                        bolt3_mae_lst.append(bolt_mae_lst_)
                    else:
                        bolt5_mae_lst.append(bolt_mae_lst_)

            a1 = np.min(np.array(bolt1_mae_lst), axis=0)
            a3 = np.min(np.array(bolt3_mae_lst), axis=0)
            a5 = np.min(np.array(bolt5_mae_lst), axis=0)


            summary = pd.DataFrame({'All Ens': all_ens_mae_lst,
                                    'Exp(1)': np.array(bolt1_mae_lst).T[np.argmin(a1)],
                                    'Exp(3)': np.array(bolt3_mae_lst).T[np.argmin(a3)],
                                    'Exp(5)': np.array(bolt5_mae_lst).T[np.argmin(a5)]})
            
            summary.to_csv(f'inference/{model_name}_{datasets[d]}_MAE_summary.csv')
            summary.describe()
        
        else:
            print(f"{datasets[d]}의 {model_name}은 아직 학습되지 않았으므로 해당 프로세스 유예")

--- 파일 검색 디버깅 ---
검색 중인 폴더: result/TEM/test/
폴더 안의 모든 파일: ['trTFMLP_TEM_mase_pred.csv', 'trTFMLP_TEM_mse_pred.csv', 'trTFMLP_TEM_smape_pred.csv', 'trTFTF_TEM_mae_pred.csv', 'trTFTF_TEM_smape_pred.csv', 'trTFMLP_TEM_mae_pred.csv', 'trTFTF_TEM_mase_pred.csv', 'trTFTF_TEM_mape_pred.csv', 'trTFMLP_TEM_mape_pred.csv', 'trTFTF_TEM_mse_pred.csv']
찾고 있는 파일 시작 부분(prefix): 'trTFTF_TEM'
--- 디버깅 끝 ---
--- 파일 검색 디버깅 ---
검색 중인 폴더: result/TEM/test/
폴더 안의 모든 파일: ['trTFMLP_TEM_mase_pred.csv', 'trTFMLP_TEM_mse_pred.csv', 'trTFMLP_TEM_smape_pred.csv', 'trTFTF_TEM_mae_pred.csv', 'trTFTF_TEM_smape_pred.csv', 'trTFMLP_TEM_mae_pred.csv', 'trTFTF_TEM_mase_pred.csv', 'trTFTF_TEM_mape_pred.csv', 'trTFMLP_TEM_mape_pred.csv', 'trTFTF_TEM_mse_pred.csv']
찾고 있는 파일 시작 부분(prefix): 'trTFMLP_TEM'
--- 디버깅 끝 ---
TEM의 trTFLSTM은 아직 학습되지 않았으므로 해당 프로세스 유예
--- 파일 검색 디버깅 ---
검색 중인 폴더: result/SOL/test/
폴더 안의 모든 파일: ['trTFMLP_SOL_mae_pred.csv', 'trTFMLP_SOL_mse_pred.csv', 'trTFTF_SOL_mse_pred.csv', 'trTFMLP_SOL_mase_pred.csv', 't

In [5]:
df = pd.DataFrame(columns = ["dataset", "model", "loss", "All Ens", "Exp(1)", "Exp(3)", "Exp(5)"])

for data in datasets:
    for model_name in model_names:
        if agg_nonskip(model_name, data):
            rec_rmse = []
            rec_mae = []

            df_rmse = pd.read_csv(f'inference/{model_name}_{data}_RMSE_summary.csv').iloc[:,1:].describe().T.iloc[:,1:3].round(5)
            df_rmse['mean(std)'] = df_rmse.apply(
                lambda r: f"{r['mean']:.3f} ({r['std']:.4f})", axis=1
            )

            df_mae = pd.read_csv(f'inference/{model_name}_{data}_MAE_summary.csv').iloc[:,1:].describe().T.iloc[:,1:3].round(5)
            df_mae['mean(std)'] = df_mae.apply(
                lambda r: f"{r["mean"]:.3f} ({r["std"]:.4f})", axis = 1
            )

            rec_rmse.append(df_rmse.loc[:,'mean(std)'])
            rec_mae.append(df_mae.loc[:, "mean(std)"])

            dfff_rmse = pd.DataFrame(rec_rmse).T
            dfff_mae = pd.DataFrame(rec_mae).T

            dfff_rmse.columns = [data]
            dfff_mae.columns = [data]

            dfff_rmse = dfff_rmse.T.reset_index().rename({"index": "dataset"}, axis = 1).assign(model = model_name).set_index(["dataset", "model"]).reset_index().assign(loss = "RMSE")
            dfff_mae = dfff_mae.T.reset_index().rename({"index": "dataset"}, axis = 1).assign(model = model_name).set_index(["dataset", "model"]).reset_index().assign(loss = "MAE")

            df = pd.concat([df, dfff_rmse, dfff_mae], axis = 0)

df = df.assign(dataset = pd.Categorical(df.dataset, categories = datasets, ordered = True))            

In [6]:
value_cols = ['All Ens', 'Exp(1)', 'Exp(3)', 'Exp(5)']

df_long = df.melt(
    id_vars=['model', 'dataset', 'loss'],
    value_vars=value_cols,
    var_name='experiment',
    value_name='value'
)

df_final = (
    df_long
    .pivot_table(
        index=['dataset', 'model'],
        columns=['experiment', 'loss'],
        values='value',
        aggfunc='first'
    )
)

df_final = df_final.reindex(
    columns=pd.MultiIndex.from_product(
        [['All Ens', 'Exp(1)', 'Exp(3)', 'Exp(5)'],
        ['RMSE', 'MAE']]
    )
)

df_final.to_csv("aggregated_result.csv")
df_final

/tmp/ipykernel_49431/672891011.py:12: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(


All Ens                          Exp(1)  \
                           RMSE             MAE            RMSE   
dataset model                                                     
TEM     trTFMLP  3.640 (0.0049)  2.878 (0.0031)  3.645 (0.0117)   
        trTFTF   4.030 (0.0343)  3.172 (0.0229)  3.965 (0.0198)   
SOL     trTFMLP  0.985 (0.0024)  0.607 (0.0013)  0.916 (0.0060)   
        trTFTF   1.036 (0.0012)  0.595 (0.0004)  0.922 (0.0014)   
ELE     trTFMLP  0.641 (0.0005)  0.517 (0.0006)  0.642 (0.0004)   
AIR     trTFMLP  6.636 (0.0090)  5.120 (0.0132)  6.619 (0.0146)   
        trTFTF   6.881 (0.0121)  5.328 (0.0143)  6.855 (0.0112)   
WID     trTFMLP  1.669 (0.0014)  1.255 (0.0011)  1.681 (0.0125)   
        trTFTF   1.737 (0.0044)  1.321 (0.0036)  1.737 (0.0034)   
MET     trTFMLP  1.980 (0.0038)  1.671 (0.0018)  1.965 (0.0038)   
coin    trTFMLP  3.643 (0.0026)  1.392 (0.0044)  3.634 (0.0022)   
        trTFTF   3.881 (0.0181)  1.697 (0.0297)  3.848 (0.0130)   

                                         Exp(3)                  \
                            MAE            RMSE             MAE   
dataset model                                                     
TEM     trTFMLP  2.886 (0.0062)  3.630 (0.0133)  2.874 (0.0065)   
        trTFTF   3.141 (0.0131)  3.856 (0.0164)  3.077 (0.0120)   
SOL     trTFMLP  0.649 (0.0037)  0.918 (0.0061)  0.629 (0.0040)   
        trTFTF   0.634 (0.0007)  0.921 (0.0015)  0.615 (0.0005)   
ELE     trTFMLP  0.518 (0.0007)  0.641 (0.0004)  0.518 (0.0006)   
AIR     trTFMLP  5.114 (0.0215)  6.614 (0.0089)  5.089 (0.0124)   
        trTFTF   5.316 (0.0144)  6.839 (0.0136)  5.311 (0.0140)   
WID     trTFMLP  1.258 (0.0032)  1.680 (0.0142)  1.258 (0.0035)   
        trTFTF   1.320 (0.0032)  1.733 (0.0034)  1.319 (0.0032)   
MET     trTFMLP  1.675 (0.0013)  1.958 (0.0036)  1.674 (0.0011)   
coin    trTFMLP  1.382 (0.0037)  3.632 (0.0022)  1.369 (0.0033)   
        trTFTF   1.669 (0.0227)  3.829 (0.0104)  1.640 (0.0143)   

                         Exp(5)                  
                           RMSE             MAE  
dataset model                                    
TEM     trTFMLP  3.626 (0.0138)  2.869 (0.0068)  
        trTFTF   3.819 (0.0138)  3.048 (0.0110)  
SOL     trTFMLP  0.919 (0.0059)  0.619 (0.0043)  
        trTFTF   0.919 (0.0017)  0.605 (0.0004)  
ELE     trTFMLP  0.641 (0.0004)  0.518 (0.0006)  
AIR     trTFMLP  6.615 (0.0086)  5.073 (0.0115)  
        trTFTF   6.832 (0.0150)  5.306 (0.0142)  
WID     trTFMLP  1.680 (0.0158)  1.258 (0.0038)  
        trTFTF   1.730 (0.0034)  1.318 (0.0032)  
MET     trTFMLP  1.952 (0.0033)  1.674 (0.0010)  
coin    trTFMLP  3.632 (0.0022)  1.368 (0.0033)  
        trTFTF   3.824 (0.0109)  1.634 (0.0142)